In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('/opt/workspace')\n",
    "\n",
    "from connectors.clickhouse_client import ClickHouseClient\n",
    "from config.settings import clickhouse_config"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client = ClickHouseClient()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.execute_query(\"CREATE DATABASE IF NOT EXISTS bronze\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "bronze_snapshot_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS bronze.snapshot_raw\n",
    "(\n",
    "    ref_date Date,\n",
    "    table_name String,\n",
    "    primary_key String,\n",
    "    row_hash String,\n",
    "    data String,\n",
    "    ingestion_timestamp DateTime DEFAULT now()\n",
    ")\n",
    "ENGINE = MergeTree()\n",
    "PARTITION BY toYYYYMM(ref_date)\n",
    "ORDER BY (ref_date, table_name, primary_key)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(bronze_snapshot_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "bronze_control_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS bronze.ingestion_control\n",
    "(\n",
    "    ref_date Date,\n",
    "    table_name String,\n",
    "    schema_name String,\n",
    "    row_count UInt64,\n",
    "    ingestion_start DateTime,\n",
    "    ingestion_end DateTime,\n",
    "    status String,\n",
    "    error_message String\n",
    ")\n",
    "ENGINE = MergeTree()\n",
    "PARTITION BY toYYYYMM(ref_date)\n",
    "ORDER BY (ref_date, table_name)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(bronze_control_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.execute_query(\"CREATE DATABASE IF NOT EXISTS silver\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "silver_delta_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS silver.delta_events\n",
    "(\n",
    "    ref_date Date,\n",
    "    table_name String,\n",
    "    primary_key String,\n",
    "    operation_type Enum8('INSERT' = 1, 'UPDATE' = 2, 'DELETE' = 3),\n",
    "    row_hash_before String,\n",
    "    row_hash_after String,\n",
    "    data_before String,\n",
    "    data_after String,\n",
    "    detected_at DateTime DEFAULT now()\n",
    ")\n",
    "ENGINE = MergeTree()\n",
    "PARTITION BY toYYYYMM(ref_date)\n",
    "ORDER BY (ref_date, table_name, primary_key, operation_type)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(silver_delta_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "silver_state_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS silver.current_state\n",
    "(\n",
    "    table_name String,\n",
    "    primary_key String,\n",
    "    row_hash String,\n",
    "    data String,\n",
    "    first_seen_date Date,\n",
    "    last_seen_date Date,\n",
    "    is_active UInt8,\n",
    "    updated_at DateTime DEFAULT now()\n",
    ")\n",
    "ENGINE = ReplacingMergeTree(updated_at)\n",
    "PARTITION BY table_name\n",
    "ORDER BY (table_name, primary_key)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(silver_state_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.execute_query(\"CREATE DATABASE IF NOT EXISTS gold\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "gold_metrics_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS gold.daily_change_metrics\n",
    "(\n",
    "    ref_date Date,\n",
    "    table_name String,\n",
    "    total_inserts UInt64,\n",
    "    total_updates UInt64,\n",
    "    total_deletes UInt64,\n",
    "    total_active_records UInt64,\n",
    "    calculated_at DateTime DEFAULT now()\n",
    ")\n",
    "ENGINE = SummingMergeTree()\n",
    "PARTITION BY toYYYYMM(ref_date)\n",
    "ORDER BY (ref_date, table_name)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(gold_metrics_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "gold_quality_ddl = \"\"\"\n",
    "CREATE TABLE IF NOT EXISTS gold.data_quality_metrics\n",
    "(\n",
    "    ref_date Date,\n",
    "    table_name String,\n",
    "    null_count UInt64,\n",
    "    duplicate_count UInt64,\n",
    "    total_records UInt64,\n",
    "    quality_score Float64,\n",
    "    calculated_at DateTime DEFAULT now()\n",
    ")\n",
    "ENGINE = ReplacingMergeTree(calculated_at)\n",
    "PARTITION BY toYYYYMM(ref_date)\n",
    "ORDER BY (ref_date, table_name)\n",
    "SETTINGS index_granularity = 8192\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(gold_quality_ddl)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "result = client.execute_query_with_result(\"SHOW DATABASES\")\n",
    "result.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.close()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.14"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}